<a href="https://colab.research.google.com/github/tokoblotongan/IPTV/blob/master/Xtreme_codes_to_m3u.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import datetime
import random
import re
import ipywidgets as widgets
from IPython.display import display, HTML
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote, urlparse
import csv
import os

requests.packages.urllib3.disable_warnings()

# ============================================================
#  IPTV ULTIMATE TOOL v9
#  Fix: MAC converter pakai threading untuk resolve URL
#       + progress real-time + opsi skip resolve
# ============================================================

display(HTML("""
<div style="font-family:monospace;background:#1e293b;color:#f8fafc;
            padding:14px 20px;border-radius:10px;margin-bottom:8px;">
  <span style="font-size:18px;font-weight:700;">⚡ IPTV Ultimate Tool v9</span><br>
  <span style="font-size:12px;color:#94a3b8;">
    Scanner + Converter → M3U &nbsp;|&nbsp; Deteksi & Filter XXX/Adult
  </span>
</div>
"""))

# ====================== MODE ======================
mode = widgets.RadioButtons(
    options=['Xtream', 'MAC', 'Raw URL (M3U/M3U8)'],
    value='Xtream', description='Mode:',
    layout=widgets.Layout(width='600px')
)
server = widgets.Text(description='Server:', placeholder='http://domain:port',
                      layout=widgets.Layout(width='500px'))
user   = widgets.Text(description='User:',   layout=widgets.Layout(width='400px'))
passwd = widgets.Password(description='Pass:', layout=widgets.Layout(width='400px'))
portal  = widgets.Text(description='Portal:', placeholder='http://domain:port/c/',
                       layout=widgets.Layout(width='500px'))
mac_inp = widgets.Text(description='MAC:', placeholder='00:1A:79:XX:XX:XX',
                       layout=widgets.Layout(width='400px'))
raw_url_inp = widgets.Text(
    description='M3U URL:', placeholder='http://domain/playlist.m3u atau .m3u8',
    layout=widgets.Layout(width='600px')
)
raw_url_note = widgets.HTML(
    value='<span style="color:#6b7280;font-size:12px;">'
          '⚡ Masukkan URL file .m3u/.m3u8 — diparse & dicek channel.</span>'
)

# ====================== TABS AKSI ======================
action_tab = widgets.Tab()

# ---- SCANNER UI ----
threads_slider = widgets.IntSlider(
    value=3, min=1, max=10, description='Threads:',
    layout=widgets.Layout(width='500px')
)
scan_mode = widgets.RadioButtons(
    options=['Sample (cepat)', 'FULL (semua channel, lambat)'],
    value='Sample (cepat)', description='Scan:',
    layout=widgets.Layout(width='500px')
)
sample_size_slider = widgets.IntSlider(
    value=10, min=5, max=30, description='Sample/cat:',
    layout=widgets.Layout(width='500px')
)
export_csv_chk = widgets.Checkbox(description="Export hasil ke CSV?", value=False)
btn_scan       = widgets.Button(description="🚀 RUN SCAN", button_style='success',
                                layout=widgets.Layout(width='100%'))
progress       = widgets.IntProgress(value=0, min=0, max=100, description='Progress:',
                                     bar_style='info', layout=widgets.Layout(width='100%'))
status_label   = widgets.Label(value="✅ Siap")
out_summary  = widgets.Output()
out_channels = widgets.Output()
out_xxx      = widgets.Output()
result_tab   = widgets.Tab(children=[out_summary, out_channels, out_xxx])
result_tab.set_title(0, '📊 Summary')
result_tab.set_title(1, '📋 Semua Channel')
result_tab.set_title(2, '🔞 XXX/Adult')
scanner_box = widgets.VBox([
    threads_slider, scan_mode, sample_size_slider, export_csv_chk,
    btn_scan, progress, status_label, result_tab
])

# ---- CONVERTER UI ----
conv_type_live   = widgets.Checkbox(value=True,  description='Live TV')
conv_type_xxx    = widgets.Checkbox(value=False, description='🔞 XXX/Adult')
conv_type_vod    = widgets.Checkbox(value=False, description='VOD (Film)')
conv_type_series = widgets.Checkbox(value=False, description='Series')
conv_xxx_note    = widgets.HTML(
    value='<span style="color:#dc2626;font-size:12px;font-weight:600;">'
          '⚠️  XXX/Adult dideteksi otomatis. Centang untuk disertakan.</span>'
)
conv_type_box = widgets.VBox([
    widgets.HTML('<b style="font-family:monospace;font-size:13px;">Konten:</b>'),
    widgets.HBox([conv_type_live, conv_type_xxx, conv_type_vod, conv_type_series]),
    conv_xxx_note
])

# --- MAC resolve mode ---
mac_resolve_mode = widgets.RadioButtons(
    options=[
        'Resolve URL (akurat, lebih lambat)',
        'Gunakan CMD langsung (cepat, mungkin perlu player khusus)',
        'Tanpa resolve — simpan sebagai URL mentah'
    ],
    value='Gunakan CMD langsung (cepat, mungkin perlu player khusus)',
    description='MAC Mode:',
    layout=widgets.Layout(width='700px')
)
mac_resolve_note = widgets.HTML(
    value='''<div style="font-family:monospace;font-size:12px;background:#1e3a5f;
             color:#bfdbfe;padding:8px 12px;border-radius:6px;margin:4px 0;">
  <b>Mode MAC Converter:</b><br>
  • <b>Resolve URL</b>     — request ke server 1x per channel → akurat tapi lambat untuk 10.000+ ch<br>
  • <b>CMD langsung</b>    — ambil URL dari field cmd (sudah ada di data) → cepat, cocok untuk kebanyakan player<br>
  • <b>URL mentah</b>      — simpan cmd as-is tanpa modifikasi → paling cepat
</div>'''
)
# Thread untuk resolve MAC
mac_resolve_threads = widgets.IntSlider(
    value=10, min=1, max=30, description='Threads:',
    layout=widgets.Layout(width='500px')
)
mac_resolve_threads_label = widgets.HTML(
    value='<span style="font-size:12px;color:#6b7280;">Thread untuk resolve URL (hanya berlaku di mode "Resolve URL")</span>'
)

conv_fmt = widgets.RadioButtons(
    options=['.ts (default)', '.m3u8 (HLS)', '.mp4'],
    value='.ts (default)', description='Format:',
    layout=widgets.Layout(width='400px')
)
conv_epg = widgets.Text(
    description='EPG URL:', placeholder='(opsional) http://epg.domain.com/epg.xml',
    layout=widgets.Layout(width='600px')
)
conv_filter = widgets.Text(
    description='Filter grup:', placeholder='(opsional) sport,hbo — kosong = semua',
    layout=widgets.Layout(width='600px')
)
conv_fname = widgets.Text(
    description='Nama file:', value='playlist.m3u',
    layout=widgets.Layout(width='400px')
)
btn_convert   = widgets.Button(description="🎬 CONVERT → M3U", button_style='warning',
                               layout=widgets.Layout(width='100%'))
conv_progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:',
                                    bar_style='warning', layout=widgets.Layout(width='100%'))
conv_progress_detail = widgets.IntProgress(
    value=0, min=0, max=100, description='Resolve:',
    bar_style='info', layout=widgets.Layout(width='100%')
)
conv_status   = widgets.Label(value="✅ Siap convert")
out_conv      = widgets.Output()

converter_box = widgets.VBox([
    conv_type_box,
    mac_resolve_mode, mac_resolve_note,
    mac_resolve_threads, mac_resolve_threads_label,
    conv_fmt, conv_epg, conv_filter, conv_fname,
    btn_convert, conv_progress, conv_progress_detail, conv_status, out_conv
])

action_tab.children = [scanner_box, converter_box]
action_tab.set_title(0, '🔍 Scanner')
action_tab.set_title(1, '💾 Convert → M3U')

display(mode, server, user, passwd, portal, mac_inp, raw_url_inp, raw_url_note, action_tab)

# ====================== KONSTANTA ======================
categories = {
    "HBO":   ["hbo"],
    "TNT":   ["tnt"],
    "SPORT": ["sport", "bein", "espn", "mola", "liga", "premier"],
    "MOVIE": ["movie", "cinema", "film", "paramount", "hulu"],
    "KIDS":  ["kids", "cartoon", "disney", "nick", "toon"],
    "NEWS":  ["news", "cnn", "bbc", "fox"],
    "PPV":   ["ppv", "event", "ufc", "wwe", "boxing", "fight"]
}
weights = {
    "SPORT": 1.40, "PPV": 1.40, "HBO": 1.25, "TNT": 1.20,
    "MOVIE": 1.10, "KIDS": 0.80, "NEWS": 0.70
}
XXX_KEYWORDS = [
    "xxx", "adult", "porn", "sex", "erotic", "18+", "xvideos", "xnxx",
    "brazzers", "playboy", "penthouse", "hustler", "nude", "naked",
    "milf", "anal", "hardcore", "softcore", "hentai",
    "redtube", "pornhub", "youporn", "spankbang",
    "granny", "mature", "fetish", "bdsm", "shemale",
    "big tits", "big ass", "cam4", "chaturbate",
]
MAG_UA = (
    "Mozilla/5.0 (QtEmbedded; U; Linux; C) AppleWebKit/533.3"
    " (KHTML, like Gecko) MAG200 stbapp ver: 2 rev: 250 Safari/533.3"
)

# ====================== HELPERS ======================
def is_xxx(name, group=""):
    combined = (str(name) + " " + str(group)).lower()
    return any(k in combined for k in XXX_KEYWORDS)

def detect_category(name, group=""):
    if is_xxx(name, group): return "XXX"
    n = str(name).lower()
    for cat, keys in categories.items():
        if any(k in n for k in keys): return cat
    return "OTHER"

def validate_url(url):
    try:
        r = urlparse(url.strip())
        return r.scheme in ("http", "https") and bool(r.netloc)
    except:
        return False

def validate_mac(mac_addr):
    return bool(mac_addr and re.match(
        r'^([0-9A-Fa-f]{2}[:\-]){5}([0-9A-Fa-f]{2})$', mac_addr.strip()
    ))

def fmt_expiry(exp):
    try:
        if exp and str(exp).isdigit():
            return datetime.datetime.fromtimestamp(int(exp)).strftime('%Y-%m-%d %H:%M')
        return str(exp) if exp else "Unlimited"
    except:
        return str(exp)

def get_stream_ext(fmt_val):
    if "m3u8" in fmt_val: return "m3u8"
    if "mp4"  in fmt_val: return "mp4"
    return "ts"

# ====================== STREAM CHECK ======================
def is_stream_alive(url):
    if not url or "://" not in str(url): return False
    headers = {"User-Agent": MAG_UA, "Connection": "close", "Accept": "*/*"}
    try:
        with requests.get(url, timeout=12, stream=True, headers=headers, verify=False) as r:
            if r.status_code not in (200, 206): return False
            ctype = r.headers.get('Content-Type', '').lower()
            if any(x in ctype for x in ["text/html", "text/plain", "application/json"]):
                return False
            return len(r.raw.read(8192)) > 512
    except:
        return False

# ====================== PARSE M3U ======================
def parse_m3u(content):
    channels, lines, meta = [], content.splitlines(), {}
    for line in lines:
        line = line.strip()
        if not line: continue
        if line.startswith("#EXTINF"):
            meta = {"name": "Unknown", "group": "OTHER"}
            tvg  = re.search(r'tvg-name="([^"]*)"', line)
            if tvg: meta["name"] = tvg.group(1).strip()
            grp  = re.search(r'group-title="([^"]*)"', line)
            if grp: meta["group"] = grp.group(1).strip()
            if meta["name"] == "Unknown":
                parts = line.split(",", 1)
                if len(parts) == 2: meta["name"] = parts[1].strip()
        elif line.startswith(("http://", "https://", "rtmp")):
            channels.append({
                "name":  meta.get("name", "Unknown") if meta else "Unknown",
                "group": meta.get("group", "OTHER")  if meta else "OTHER",
                "url":   line
            })
            meta = {}
    return channels

def fetch_m3u_url(url):
    headers = {"User-Agent": "VLC/3.0.18 LibVLC/3.0.18", "Accept": "*/*"}
    resp    = requests.get(url.strip(), timeout=20, headers=headers, verify=False)
    resp.raise_for_status()
    content = resp.text
    if not content.strip().startswith("#EXTM3U") and "extinf" not in content.lower()[:500]:
        raise ValueError("File bukan format M3U/M3U8 yang valid.")
    return parse_m3u(content)

# ====================== MAC AUTH ======================
def mac_auth(portal_val, mac_clean):
    session = requests.Session()
    session.headers.update({
        "User-Agent": MAG_UA, "X-Requested-With": "XMLHttpRequest",
        "Referer": f"{portal_val}portal.php"
    })
    session.cookies.update({"mac": mac_clean, "stb_lang": "en", "timezone": "Europe/Amsterdam"})
    h     = session.get(
        f"{portal_val}portal.php?type=stb&action=handshake&JsHttpRequest=1-xml",
        timeout=12, verify=False
    ).json()
    js_h  = h.get("js")
    token = js_h.get("token") if isinstance(js_h, dict) else None
    if token:
        session.headers.update({"Authorization": f"Bearer {token}"})
    return session, token

def get_mac_channels(portal_val, session):
    try:
        resp   = session.get(
            f"{portal_val}portal.php?type=itv&action=get_all_channels&JsHttpRequest=1-xml",
            timeout=15, verify=False
        ).json()
        js_val = resp.get("js")
        ch     = js_val.get("data", []) if isinstance(js_val, dict) else []
        if ch:
            print(f"  ✅ get_all_channels: {len(ch)} channel")
            return ch
    except:
        pass
    print("  ⚠️  Coba get_ordered_list (paginasi)...")
    ch_all, page = [], 1
    while True:
        try:
            resp   = session.get(
                f"{portal_val}portal.php?type=itv&action=get_ordered_list"
                f"&genre=*&p={page}&JsHttpRequest=1-xml",
                timeout=15, verify=False
            ).json()
            js_val = resp.get("js")
            if not isinstance(js_val, dict): break
            data  = js_val.get("data", [])
            total = int(js_val.get("total_items", 0))
            if not data: break
            ch_all.extend(data)
            print(f"  Page {page}: +{len(data)} (total: {len(ch_all)}/{total})")
            if total and len(ch_all) >= total: break
            if len(data) < 14: break
            page += 1
        except:
            break
    return ch_all

# ====================== RESOLVE MAC URL ======================
def resolve_mac_url(portal_val, cmd, session):
    """Resolve satu channel URL via create_link."""
    try:
        cmd_encoded = quote(str(cmd), safe=':/?&= #')
        r   = session.get(
            f"{portal_val}portal.php?type=itv&action=create_link"
            f"&cmd={cmd_encoded}&JsHttpRequest=1-xml",
            timeout=8, verify=False
        )
        js_val = r.json().get("js")
        if not isinstance(js_val, dict): return None
        raw = str(js_val.get("cmd", "") or js_val.get("url", "")).strip()
        for part in reversed(raw.split()):
            if "://" in part: return part
        return None
    except:
        return None

def extract_cmd_url(cmd):
    """
    Ekstrak URL langsung dari field cmd tanpa request ke server.
    Contoh cmd: 'ffmpeg http://domain/live/xxx.ts' → 'http://domain/live/xxx.ts'
    Atau cmd sudah berupa URL langsung.
    """
    if not cmd: return ""
    cmd = str(cmd).strip()
    # Cari http/https/rtmp di dalam string
    for part in reversed(cmd.split()):
        if part.startswith(("http://", "https://", "rtmp://", "rtmpe://")):
            return part
    # Kalau tidak ada prefix, coba cmd langsung sebagai URL
    if cmd.startswith(("http://", "https://", "rtmp://")):
        return cmd
    return cmd  # kembalikan apa adanya

# ====================== THREADED MAC RESOLVE ======================
def resolve_mac_channels_threaded(portal_val, session, ch_list, n_threads,
                                   progress_bar, status_lbl, resolve_mode):
    """
    Resolve URL semua channel MAC secara paralel.
    resolve_mode:
      'resolve'  → pakai create_link (lambat, akurat)
      'cmd'      → ekstrak URL dari field cmd (cepat)
      'raw'      → simpan cmd as-is
    """
    total   = len(ch_list)
    results = [None] * total  # preserve order

    progress_bar.max   = total
    progress_bar.value = 0

    def _resolve_one(idx_obj):
        idx, obj = idx_obj
        grp  = (obj.get("genre_name") or obj.get("category_name") or
                str(obj.get("tv_genre_id", "")) or "MAC TV")
        name = obj.get("name", "Unknown")
        cmd  = obj.get("cmd", "")

        if resolve_mode == "resolve":
            url = resolve_mac_url(portal_val, cmd, session) or extract_cmd_url(cmd)
        elif resolve_mode == "cmd":
            url = extract_cmd_url(cmd)
        else:  # raw
            url = str(cmd).strip() if cmd else ""

        return idx, {
            "name": name, "group": grp,
            "logo": obj.get("logo", ""),
            "epg_id": obj.get("xmltv_id", ""),
            "num": obj.get("number", ""),
            "stream_id": "", "url": url, "url_tpl": url
        }

    done = 0
    with ThreadPoolExecutor(max_workers=n_threads) as executor:
        futures = {executor.submit(_resolve_one, (i, obj)): i
                   for i, obj in enumerate(ch_list)}
        for future in as_completed(futures):
            try:
                idx, entry = future.result()
                results[idx] = entry
            except:
                pass
            done += 1
            progress_bar.value = done
            if done % 50 == 0 or done == total:
                status_lbl.value = f"🔗 Resolve: {done}/{total} channel..."

    return [r for r in results if r is not None]

# ====================== XTREAM FETCH ======================
def xtream_get_live(base, u, p, ext):
    url  = f"{base}/player_api.php?username={u}&password={p}&action=get_live_streams"
    data = requests.get(url, timeout=20, verify=False).json()
    cats = {}
    try:
        curl = f"{base}/player_api.php?username={u}&password={p}&action=get_live_categories"
        cats = {str(c["category_id"]): c["category_name"]
                for c in requests.get(curl, timeout=15, verify=False).json()}
    except:
        pass
    out = []
    for ch in data:
        cid = str(ch.get("category_id", ""))
        out.append({
            "name":      ch.get("name", "Unknown"),
            "group":     cats.get(cid) or ch.get("category_name", "Live TV"),
            "logo":      ch.get("stream_icon", ""),
            "epg_id":    ch.get("epg_channel_id", ""),
            "num":       ch.get("num", ""),
            "stream_id": ch.get("stream_id"),
            "url_tpl":   f"{base}/live/{u}/{p}/{{sid}}.{ext}"
        })
    return out

def xtream_get_vod(base, u, p, ext):
    url  = f"{base}/player_api.php?username={u}&password={p}&action=get_vod_streams"
    data = requests.get(url, timeout=30, verify=False).json()
    cats = {}
    try:
        curl = f"{base}/player_api.php?username={u}&password={p}&action=get_vod_categories"
        cats = {str(c["category_id"]): c["category_name"]
                for c in requests.get(curl, timeout=15, verify=False).json()}
    except:
        pass
    out = []
    for v in data:
        cid = str(v.get("category_id", ""))
        out.append({
            "name":      v.get("name", "Unknown"),
            "group":     cats.get(cid, "VOD"),
            "logo":      v.get("stream_icon", ""),
            "epg_id":    "",
            "num":       "",
            "stream_id": v.get("stream_id"),
            "url_tpl":   f"{base}/movie/{u}/{p}/{{sid}}.{ext}"
        })
    return out

def xtream_get_series(base, u, p):
    url   = f"{base}/player_api.php?username={u}&password={p}&action=get_series"
    sdata = requests.get(url, timeout=30, verify=False).json()
    cats  = {}
    try:
        curl = f"{base}/player_api.php?username={u}&password={p}&action=get_series_categories"
        cats = {str(c["category_id"]): c["category_name"]
                for c in requests.get(curl, timeout=15, verify=False).json()}
    except:
        pass
    out = []
    for s in sdata:
        sid  = s.get("series_id")
        name = s.get("name", "Unknown")
        cid  = str(s.get("category_id", ""))
        grp  = cats.get(cid, "Series")
        logo = s.get("cover", "")
        try:
            ep_data = requests.get(
                f"{base}/player_api.php?username={u}&password={p}"
                f"&action=get_series_info&series_id={sid}",
                timeout=15, verify=False
            ).json()
            for season_num, eps in ep_data.get("episodes", {}).items():
                for ep in eps:
                    ext_ep = ep.get("container_extension", "mp4")
                    title  = ep.get("title", f"S{season_num}E{ep.get('episode_num','?')}")
                    out.append({
                        "name": f"{name} — {title}", "group": grp,
                        "logo": logo, "epg_id": "", "num": "",
                        "stream_id": ep.get("id"),
                        "url_tpl": f"{base}/series/{u}/{p}/{{sid}}.{ext_ep}"
                    })
        except:
            out.append({
                "name": name, "group": grp, "logo": logo,
                "epg_id": "", "num": "", "stream_id": sid,
                "url_tpl": f"{base}/series/{u}/{p}/{{sid}}.mp4"
            })
    return out

# ====================== BUILD M3U ======================
def build_m3u_text(entries, epg_url="", filters=None, include_xxx=False):
    lines          = ['#EXTM3U' + (f' x-tvg-url="{epg_url}"' if epg_url else "")]
    skipped_xxx    = 0
    skipped_filter = 0
    for e in entries:
        grp  = e.get("group", "")
        name = e.get("name", "")
        if is_xxx(name, grp):
            if not include_xxx:
                skipped_xxx += 1
                continue
        if filters and not any(f in grp.lower() or f in name.lower() for f in filters):
            skipped_filter += 1
            continue
        sid = e.get("stream_id")
        if sid == "":
            url = e.get("url_tpl", e.get("url", ""))
        else:
            url = e["url_tpl"].format(sid=sid) if sid else e.get("url", "")
        ext_line = '#EXTINF:-1'
        if e.get("epg_id"): ext_line += f' tvg-id="{e["epg_id"]}"'
        ext_line += f' tvg-name="{name}"'
        if e.get("logo"):   ext_line += f' tvg-logo="{e["logo"]}"'
        if grp:             ext_line += f' group-title="{grp}"'
        if e.get("num"):    ext_line += f' tvg-chno="{e["num"]}"'
        ext_line += f',{name}'
        lines.append(ext_line)
        lines.append(url)
    return "\n".join(lines), skipped_xxx, skipped_filter

# ====================== WORKER (SCAN) ======================
def worker(task):
    cat, ch_obj, mode_type, s_val, u_val, p_val, portal_val, session = task
    name = str(ch_obj.get("name", "Unknown"))
    url_checked = ""
    try:
        if mode_type == "Xtream":
            sid         = ch_obj.get("stream_id")
            url_checked = f"{s_val}/live/{u_val}/{p_val}/{sid}.ts"
            alive       = is_stream_alive(url_checked)
        elif mode_type == "Raw URL (M3U/M3U8)":
            url_checked = ch_obj.get("url", "")
            alive       = is_stream_alive(url_checked)
        else:
            cmd = ch_obj.get("cmd")
            if not cmd: return cat, False, name, ""
            url_checked = resolve_mac_url(portal_val, cmd, session) or ""
            alive       = is_stream_alive(url_checked) if url_checked else False
        return cat, alive, name, url_checked
    except:
        return cat, False, name, ""

# ====================== EXPORT CSV ======================
def save_summary_csv(result, score, filename):
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(["Category", "Total", "Samples", "Alive", "Rate (%)"])
        for cat, data in result.items():
            rate = (data["alive"] / data["samples"] * 100) if data["samples"] > 0 else 0
            w.writerow([cat, data["total"], data["samples"], data["alive"], f"{rate:.2f}"])
        w.writerow([]); w.writerow(["Global Score", score])

def save_channel_csv(channel_log, filename):
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(["No", "Channel Name", "Category", "Status", "Stream URL"])
        for i, row in enumerate(channel_log, 1):
            w.writerow([i, row["name"], row["cat"], row["status"], row["url"]])

# ====================== RENDER TABLE ======================
def render_channel_table(channel_log, xxx_only=False):
    cat_colors = {
        "HBO": "#7c3aed", "TNT": "#1d4ed8", "SPORT": "#047857",
        "MOVIE": "#b45309", "KIDS": "#db2777", "NEWS": "#0369a1",
        "PPV": "#dc2626", "OTHER": "#6b7280", "XXX": "#be185d"
    }
    rows = [r for r in channel_log if r["cat"] == "XXX"] if xxx_only else channel_log
    rows_html = ""
    for i, row in enumerate(rows, 1):
        color    = cat_colors.get(row["cat"], "#6b7280")
        status   = row["status"]
        bg       = "#f0fdf4" if status == "✅ ALIVE" else "#fef2f2"
        st_color = "#16a34a" if status == "✅ ALIVE" else "#dc2626"
        rows_html += f"""
        <tr style="background:{bg};">
          <td style="padding:4px 8px;color:#6b7280;font-size:12px;">{i}</td>
          <td style="padding:4px 8px;font-weight:500;">{row['name']}</td>
          <td style="padding:4px 8px;">
            <span style="background:{color};color:#fff;padding:2px 8px;
                   border-radius:9999px;font-size:11px;font-weight:600;">{row['cat']}</span>
          </td>
          <td style="padding:4px 8px;font-weight:700;color:{st_color};">{status}</td>
        </tr>"""
    alive_count = sum(1 for r in rows if r["status"] == "✅ ALIVE")
    title = "🔞 Channel XXX/Adult" if xxx_only else "📋 Semua Channel"
    return f"""
    <div style="font-family:monospace;">
      <div style="margin-bottom:6px;font-size:14px;font-weight:700;">{title}</div>
      <div style="margin-bottom:8px;font-size:13px;">
        Total: <b>{len(rows)}</b> | <span style="color:#16a34a;font-weight:700;">✅ {alive_count}</span>
        | <span style="color:#dc2626;font-weight:700;">❌ {len(rows)-alive_count}</span>
      </div>
      <table style="border-collapse:collapse;width:100%;font-size:13px;border:1px solid #e5e7eb;">
        <thead><tr style="background:#1e293b;color:#fff;">
          <th style="padding:6px 8px;text-align:left;width:40px;">No</th>
          <th style="padding:6px 8px;text-align:left;">Channel Name</th>
          <th style="padding:6px 8px;text-align:left;width:100px;">Kategori</th>
          <th style="padding:6px 8px;text-align:left;width:100px;">Status</th>
        </tr></thead>
        <tbody>{rows_html}</tbody>
      </table>
    </div>"""

# ============================================================
#  SCANNER
# ============================================================
def run_scan(b):
    out_summary.clear_output()
    out_channels.clear_output()
    out_xxx.clear_output()
    progress.value = 0
    status_label.value = "🔄 Sedang memproses..."

    if mode.value == "Xtream":
        if not validate_url(server.value):
            status_label.value = "❌ Server URL tidak valid."
            with out_summary: print("❌ Server URL tidak valid.")
            return
    elif mode.value == "MAC":
        if not validate_url(portal.value) or not validate_mac(mac_inp.value):
            status_label.value = "❌ Portal/MAC tidak valid."
            with out_summary: print("❌ Portal atau MAC tidak valid.")
            return
    else:
        if not validate_url(raw_url_inp.value.strip()):
            status_label.value = "❌ M3U URL tidak valid."
            with out_summary: print("❌ M3U URL tidak valid.")
            return

    with out_summary:
        print("🚀 Memulai Scan...\n")
        result = {k: {"total": 0, "alive": 0, "samples": 0} for k in categories}
        result["XXX"]   = {"total": 0, "alive": 0, "samples": 0}
        result["OTHER"] = {"total": 0, "alive": 0, "samples": 0}
        ch = []; session = None; portal_val = ""

        try:
            if mode.value == "Xtream":
                print("=== XTREAM ACCOUNT ===")
                info_url = (f"{server.value}/player_api.php"
                            f"?username={user.value}&password={passwd.value}")
                data = requests.get(info_url, timeout=10, verify=False).json()
                ui   = data.get("user_info", {})
                print(f"  Status      : {ui.get('status','?')}")
                print(f"  Expired     : {fmt_expiry(ui.get('exp_date'))}")
                print(f"  Koneksi     : {ui.get('active_cons','?')}/{ui.get('max_connections','?')}\n")
                ch = requests.get(info_url + "&action=get_live_streams",
                                  timeout=15, verify=False).json()
                print(f"  Total live  : {len(ch)} channel\n")

            elif mode.value == "MAC":
                print("=== MAC ACCOUNT ===")
                mac_clean  = mac_inp.value.strip().upper()
                portal_val = portal.value.strip().rstrip('/') + '/'
                session, token = mac_auth(portal_val, mac_clean)
                print(f"  Token    : {token[:20]}..." if token else "  ⚠️  Token tidak ditemukan")
                try:
                    prof  = session.get(
                        f"{portal_val}portal.php?type=account_info"
                        f"&action=get_main_info&JsHttpRequest=1-xml",
                        timeout=10, verify=False
                    ).json().get("js", {})
                    fname = (prof.get("fname","") + " " + prof.get("lname","")).strip()
                    if fname: print(f"  Nama     : {fname}")
                    exp = prof.get("phone","") or prof.get("end_date","")
                    if exp: print(f"  Expired  : {exp}")
                except:
                    pass
                print()
                ch = get_mac_channels(portal_val, session)
                if not ch:
                    print("❌ Tidak ada channel")
                    status_label.value = "❌ Tidak ada channel"
                    return
                print(f"  Total    : {len(ch)} channel\n")

            else:
                print("=== RAW URL (M3U/M3U8) ===")
                raw_url_val  = raw_url_inp.value.strip()
                print(f"  Mengunduh: {raw_url_val}")
                raw_channels = fetch_m3u_url(raw_url_val)
                if not raw_channels:
                    print("❌ Tidak ada channel")
                    status_label.value = "❌ M3U kosong"
                    return
                print(f"  Total    : {len(raw_channels)} channel\n")
                ch = [{"name": rc["name"], "url": rc["url"], "group": rc["group"],
                        "stream_id": None, "cmd": None} for rc in raw_channels]

        except Exception as e:
            print(f"❌ Gagal: {e}")
            status_label.value = "❌ Gagal Koneksi"
            return

        is_full   = scan_mode.value.startswith("FULL")
        samp_size = sample_size_slider.value
        all_tasks = []
        channel_log = []
        all_cats    = {**{k: [] for k in categories}, "XXX": [], "OTHER": []}

        if mode.value == "Raw URL (M3U/M3U8)":
            for obj in ch:
                cat = detect_category(obj.get("name",""), obj.get("group",""))
                all_cats[cat].append(obj)
            for cat, items in all_cats.items():
                result[cat]["total"] = len(items)
                if items:
                    to_check = items if is_full else random.sample(items, min(samp_size, len(items)))
                    result[cat]["samples"] = len(to_check)
                    for obj in to_check:
                        all_tasks.append((cat, obj, "Raw URL (M3U/M3U8)", "", "", "", "", None))
        else:
            for obj in ch:
                grp = obj.get("category_name","") or obj.get("genre_name","") or ""
                cat = detect_category(obj.get("name",""), grp)
                all_cats[cat].append(obj)
            for cat, items in all_cats.items():
                result[cat]["total"] = len(items)
                if items:
                    to_check = items if is_full else random.sample(items, min(samp_size, len(items)))
                    result[cat]["samples"] = len(to_check)
                    for obj in to_check:
                        all_tasks.append((
                            cat, obj, mode.value,
                            server.value, user.value, passwd.value,
                            portal_val if mode.value == "MAC" else server.value, session
                        ))

        if not all_tasks:
            print("⚠️  Tidak ada channel yang bisa dicek.")
            status_label.value = "⚠️ Tidak ada channel"
            return

        mode_label = "FULL" if is_full else f"Sample ({samp_size}/kategori)"
        print(f"Scan Mode : {mode_label}")
        print(f"Total     : {len(all_tasks)} channel | {threads_slider.value} threads\n")
        progress.max = len(all_tasks)

        with ThreadPoolExecutor(max_workers=threads_slider.value) as executor:
            futures = {executor.submit(worker, task): task for task in all_tasks}
            for future in as_completed(futures):
                try:
                    cat, alive, name, url_checked = future.result()
                    status_str = "✅ ALIVE" if alive else "❌ DEAD"
                    channel_log.append({"name": name, "cat": cat,
                                        "status": status_str, "url": url_checked})
                    if cat in result and alive:
                        result[cat]["alive"] += 1
                except:
                    pass
                progress.value += 1
                status_label.value = f"Progress: {progress.value}/{len(all_tasks)}"

        channel_log.sort(key=lambda x: (0 if x["status"] == "✅ ALIVE" else 1, x["name"]))

        xxx_total   = result.get("XXX", {}).get("total", 0)
        xxx_samples = result.get("XXX", {}).get("samples", 0)
        xxx_alive   = result.get("XXX", {}).get("alive", 0)

        print("\n" + "=" * 65)
        print(f"{'KATEGORI':<10} {'TOTAL':<8} {'SAMPLES':<8} {'ALIVE':<6} {'RATE':<8}")
        print("-" * 65)
        weighted_sum, total_weight = 0, 0
        for cat, data in result.items():
            if cat in ("OTHER", "XXX"): continue
            if data["samples"] > 0:
                rate          = data["alive"] / data["samples"] * 100
                w             = weights.get(cat, 1.0)
                weighted_sum += rate * w
                total_weight += w
                print(f"{cat:<10} {data['total']:<8} {data['samples']:<8} {data['alive']:<6} {rate:5.1f}%")
            else:
                print(f"{cat:<10} {data['total']:<8} {'—':<8} {'—':<6} {'N/A':>5}")
        if result["OTHER"]["samples"] > 0:
            d    = result["OTHER"]
            rate = d["alive"]/d["samples"]*100 if d["samples"] > 0 else 0
            print(f"{'OTHER':<10} {d['total']:<8} {d['samples']:<8} {d['alive']:<6} {rate:5.1f}%  (diluar score)")

        global_score = round(weighted_sum / total_weight, 2) if total_weight > 0 else 0
        print("-" * 65)
        print(f"🎯 GLOBAL SCORE : {global_score}/100")
        verdict = (
            "🏆 EXCELLENT PROVIDER" if global_score >= 82 else
            "👍 GOOD PROVIDER"      if global_score >= 68 else
            "⚠️  AVERAGE"           if global_score >= 50 else
            "❌ BELOW AVERAGE"
        )
        print(verdict)
        print()
        if xxx_total > 0:
            xxx_rate = xxx_alive / xxx_samples * 100 if xxx_samples > 0 else 0
            print("=" * 65)
            print(f"🔞 XXX/ADULT TERDETEKSI")
            print(f"   Total  : {xxx_total} channel")
            print(f"   Dicek  : {xxx_samples} | ALIVE: {xxx_alive} ({xxx_rate:.1f}%)")
            print(f"   → Tab '🔞 XXX/Adult' untuk daftar lengkap")
            print("=" * 65)
        else:
            print("✅ Tidak ada channel XXX/Adult terdeteksi.")

        if export_csv_chk.value:
            ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            f1 = os.path.join(os.getcwd(), f"scan_summary_{ts}.csv")
            f2 = os.path.join(os.getcwd(), f"scan_channels_{ts}.csv")
            save_summary_csv(result, global_score, f1)
            save_channel_csv(channel_log, f2)
            print(f"\n💾 Summary CSV : {f1}")
            print(f"💾 Channel CSV : {f2}")

        status_label.value = (
            f"✅ Selesai — Score: {global_score} | {verdict}"
            + (f" | 🔞 XXX: {xxx_total}" if xxx_total > 0 else "")
        )

    with out_channels:
        if channel_log:
            display(HTML(render_channel_table(channel_log, xxx_only=False)))
        else:
            print("Belum ada data.")
    with out_xxx:
        xxx_log = [r for r in channel_log if r["cat"] == "XXX"]
        if xxx_log:
            display(HTML(render_channel_table(channel_log, xxx_only=True)))
        else:
            display(HTML('<div style="font-family:monospace;color:#16a34a;padding:12px;">'
                         '✅ Tidak ada channel XXX/Adult terdeteksi.</div>'))

# ============================================================
#  CONVERTER → M3U
# ============================================================
def run_convert(b):
    out_conv.clear_output()
    conv_progress.value        = 0
    conv_progress_detail.value = 0
    conv_status.value = "🔄 Memproses..."

    if mode.value == "Raw URL (M3U/M3U8)":
        with out_conv:
            print("⚠️  Mode Raw URL tidak perlu diconvert — file sudah M3U.")
        conv_status.value = "⚠️ Tidak berlaku untuk Raw URL"
        return

    if mode.value == "Xtream":
        if not validate_url(server.value):
            conv_status.value = "❌ Server URL tidak valid."
            with out_conv: print("❌ Server URL tidak valid.")
            return
    else:
        if not validate_url(portal.value) or not validate_mac(mac_inp.value):
            conv_status.value = "❌ Portal/MAC tidak valid."
            with out_conv: print("❌ Portal atau MAC tidak valid.")
            return

    if not any([conv_type_live.value, conv_type_vod.value,
                conv_type_series.value, conv_type_xxx.value]):
        with out_conv: print("⚠️  Pilih minimal satu jenis konten.")
        conv_status.value = "⚠️ Pilih konten dulu"
        return

    ext         = get_stream_ext(conv_fmt.value)
    include_xxx = conv_type_xxx.value
    filters     = [f.strip().lower() for f in conv_filter.value.split(',')
                   if f.strip()] if conv_filter.value.strip() else None

    # Mapping pilihan resolve
    resolve_map = {
        'Resolve URL (akurat, lebih lambat)':                    'resolve',
        'Gunakan CMD langsung (cepat, mungkin perlu player khusus)': 'cmd',
        'Tanpa resolve — simpan sebagai URL mentah':             'raw'
    }
    resolve_mode_key = resolve_map.get(mac_resolve_mode.value, 'cmd')

    with out_conv:
        print("=" * 62)
        print(f"   CONVERT → M3U  [{mode.value}]")
        print("=" * 62 + "\n")

        konten_list = []
        if conv_type_live.value:   konten_list.append("Live TV")
        if conv_type_xxx.value:    konten_list.append("🔞 XXX")
        if conv_type_vod.value:    konten_list.append("VOD")
        if conv_type_series.value: konten_list.append("Series")
        print(f"  Konten    : {' | '.join(konten_list)}")
        print(f"  XXX/Adult : {'✅ Disertakan' if include_xxx else '🚫 Dikeluarkan'}")
        print(f"  Format    : .{ext}")
        if mode.value == "MAC":
            print(f"  MAC Mode  : {mac_resolve_mode.value}")
        print()

        all_entries = []

        try:
            if mode.value == "Xtream":
                base = server.value.strip().rstrip('/')
                u    = user.value.strip()
                p    = passwd.value.strip()

                info = requests.get(
                    f"{base}/player_api.php?username={u}&password={p}",
                    timeout=12, verify=False
                ).json()
                ui = info.get("user_info", {})
                print(f"  Status   : {ui.get('status','?')}")
                print(f"  Expired  : {fmt_expiry(ui.get('exp_date'))}")
                print(f"  Koneksi  : {ui.get('active_cons','?')}/{ui.get('max_connections','?')}\n")

                steps = sum([conv_type_live.value or conv_type_xxx.value,
                             conv_type_vod.value, conv_type_series.value])
                step  = 0

                if conv_type_live.value or conv_type_xxx.value:
                    conv_status.value = "📺 Mengambil Live TV..."
                    print("  📺 Mengambil Live TV...")
                    live       = xtream_get_live(base, u, p, ext)
                    live_norm  = [e for e in live if not is_xxx(e["name"], e["group"])]
                    live_xxx   = [e for e in live if     is_xxx(e["name"], e["group"])]
                    if conv_type_live.value:
                        all_entries.extend(live_norm)
                        print(f"     Normal : {len(live_norm)} channel")
                    if live_xxx:
                        status_str = "✅ disertakan" if include_xxx else "🚫 dikeluarkan"
                        print(f"     🔞 XXX : {len(live_xxx)} channel — {status_str}")
                        if include_xxx: all_entries.extend(live_xxx)
                    step += 1; conv_progress.value = int(step/max(steps,1)*80)

                if conv_type_vod.value:
                    conv_status.value = "🎬 Mengambil VOD..."
                    print("\n  🎬 Mengambil VOD...")
                    vod = xtream_get_vod(base, u, p, ext)
                    all_entries.extend(vod)
                    print(f"     {len(vod)} film")
                    step += 1; conv_progress.value = int(step/max(steps,1)*80)

                if conv_type_series.value:
                    conv_status.value = "🎭 Mengambil Series..."
                    print("\n  🎭 Mengambil Series (bisa lambat)...")
                    ser = xtream_get_series(base, u, p)
                    all_entries.extend(ser)
                    print(f"     {len(ser)} episode")
                    step += 1; conv_progress.value = int(step/max(steps,1)*80)

            else:  # MAC
                mac_clean  = mac_inp.value.strip().upper()
                portal_val = portal.value.strip().rstrip('/') + '/'
                session, token = mac_auth(portal_val, mac_clean)
                print(f"  Token    : {token[:20]}..." if token else "  ⚠️  Token tidak ditemukan")

                conv_status.value = "📺 Mengambil daftar channel..."
                print("  📺 Mengambil channel MAC...")
                ch = get_mac_channels(portal_val, session)
                if not ch:
                    print("❌ Tidak ada channel")
                    conv_status.value = "❌ Tidak ada channel"
                    return

                # Pisah XXX
                ch_normal = []
                ch_xxx    = []
                for obj in ch:
                    grp = (obj.get("genre_name") or obj.get("category_name") or
                           str(obj.get("tv_genre_id","")) or "MAC TV")
                    if is_xxx(obj.get("name",""), grp):
                        ch_xxx.append(obj)
                    else:
                        ch_normal.append(obj)

                print(f"  Total    : {len(ch)} channel")
                print(f"  Normal   : {len(ch_normal)} | 🔞 XXX: {len(ch_xxx)}")
                status_xxx = "✅ disertakan" if include_xxx else "🚫 dikeluarkan"
                print(f"  XXX      : {status_xxx}\n")

                # Tentukan channel yang akan diproses
                to_process = ch_normal[:]
                if include_xxx:
                    to_process.extend(ch_xxx)

                conv_progress.value = 15
                print(f"  🔗 Resolve URL untuk {len(to_process)} channel")
                print(f"     Mode   : {mac_resolve_mode.value}")
                if resolve_mode_key == 'resolve':
                    print(f"     Thread : {mac_resolve_threads.value}")
                    print(f"     ⏳ Proses ini memakan waktu untuk channel banyak...\n")
                else:
                    print(f"     ⚡ Mode cepat — tidak perlu request ke server\n")

                conv_status.value = f"🔗 Resolving {len(to_process)} channel..."
                all_entries = resolve_mac_channels_threaded(
                    portal_val, session, to_process,
                    n_threads      = mac_resolve_threads.value if resolve_mode_key == 'resolve' else 1,
                    progress_bar   = conv_progress_detail,
                    status_lbl     = conv_status,
                    resolve_mode   = resolve_mode_key
                )
                conv_progress.value = 85
                resolved_count = sum(1 for e in all_entries if e.get("url",""))
                print(f"  ✅ Selesai: {len(all_entries)} channel")
                print(f"     URL valid: {resolved_count}")

        except Exception as e:
            print(f"❌ Gagal: {e}")
            conv_status.value = "❌ Gagal"
            return

        if not all_entries:
            print("❌ Tidak ada data yang berhasil diambil.")
            conv_status.value = "❌ Tidak ada data"
            return

        print(f"\n  📝 Membangun M3U dari {len(all_entries)} entri...")
        conv_status.value = "📝 Membangun M3U..."
        m3u_text, skipped_xxx, skipped_filter = build_m3u_text(
            all_entries, epg_url=conv_epg.value.strip(),
            filters=filters, include_xxx=include_xxx
        )
        included        = len(all_entries) - skipped_xxx - skipped_filter
        conv_progress.value = 95

        fname    = conv_fname.value.strip() or "playlist.m3u"
        if not fname.endswith(".m3u"): fname += ".m3u"
        out_path = os.path.join(os.getcwd(), fname)
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(m3u_text)

        conv_progress.value        = 100
        conv_progress_detail.value = conv_progress_detail.max if conv_progress_detail.max > 0 else 0

        print(f"\n{'=' * 62}")
        print(f"  Total diproses  : {len(all_entries)}")
        if not include_xxx and skipped_xxx > 0:
            print(f"  XXX dikeluarkan : {skipped_xxx}")
        if filters and skipped_filter > 0:
            print(f"  Filter dilewati : {skipped_filter}")
        print(f"  ✅ Masuk M3U    : {included}")
        print(f"  Format          : .{ext}")
        print(f"  File            : {out_path}")
        print(f"{'=' * 62}")
        print("✅ SELESAI!")

        conv_status.value = f"✅ Selesai — {included} entri → {fname}"

        display(HTML(f"""
        <div style="margin-top:12px;padding:14px 16px;background:#fffbeb;
                    border:1px solid #fcd34d;border-radius:8px;font-family:monospace;">
          <b style="color:#92400e;font-size:14px;">✅ File M3U berhasil dibuat!</b><br><br>
          <span style="font-size:13px;">
            📁 <b>{fname}</b> — {included} channel/entri<br>
            🔞 XXX: {"disertakan ✅" if include_xxx else "dikeluarkan 🚫"}<br><br>
            <b>Cara download di Google Colab:</b><br>
            1. Klik ikon 📁 <b>Files</b> di panel kiri<br>
            2. Cari file <code>{fname}</code><br>
            3. Klik kanan → <b>Download</b><br><br>
            <i>Atau jalankan cell baru:</i><br>
            <code>from google.colab import files; files.download('{fname}')</code>
          </span>
        </div>
        """))

# ====================== BIND EVENTS ======================
btn_scan.on_click(run_scan)
btn_convert.on_click(run_convert)

def toggle_ui(change):
    m   = mode.value
    xt  = m == "Xtream"
    mac = m == "MAC"
    raw = m == "Raw URL (M3U/M3U8)"
    for w in [server, user, passwd]:
        w.layout.display = 'flex' if xt else 'none'
    for w in [portal, mac_inp]:
        w.layout.display = 'flex' if mac else 'none'
    for w in [raw_url_inp, raw_url_note]:
        w.layout.display = 'flex' if raw else 'none'
    # Tampilkan MAC resolve options hanya di mode MAC
    for w in [mac_resolve_mode, mac_resolve_note, mac_resolve_threads, mac_resolve_threads_label]:
        w.layout.display = 'flex' if mac else 'none'
    if mac:
        if threads_slider.value > 5: threads_slider.value = 3
        threads_slider.max = 5
    else:
        threads_slider.max = 10

def toggle_sample(change):
    sample_size_slider.layout.display = (
        'none' if scan_mode.value.startswith("FULL") else 'flex'
    )

def toggle_resolve_threads(change):
    is_resolve = mac_resolve_mode.value.startswith('Resolve')
    mac_resolve_threads.layout.display       = 'flex' if is_resolve else 'none'
    mac_resolve_threads_label.layout.display = 'flex' if is_resolve else 'none'

mode.observe(toggle_ui, names='value')
scan_mode.observe(toggle_sample, names='value')
mac_resolve_mode.observe(toggle_resolve_threads, names='value')
toggle_ui(None)
toggle_sample(None)
toggle_resolve_threads(None)

print("✅ IPTV Ultimate Tool v9 siap!")
print()
print("  🔍 Scanner  : Xtream / MAC / Raw M3U → score, alive/dead, deteksi XXX")
print()
print("  💾 Converter: Xtream / MAC → .m3u")
print("     Konten   : Live | 🔞 XXX | VOD | Series")
print("     MAC Mode : 3 pilihan kecepatan resolve URL")
print("       • CMD langsung  → ⚡ cepat (rekomen untuk 10.000+ channel)")
print("       • Resolve URL   → akurat tapi lambat (gunakan thread tinggi)")
print("       • URL mentah    → paling cepat")

RadioButtons(description='Mode:', layout=Layout(width='600px'), options=('Xtream', 'MAC', 'Raw URL (M3U/M3U8)'…

Text(value='', description='Server:', layout=Layout(width='500px'), placeholder='http://domain:port')

Text(value='', description='User:', layout=Layout(width='400px'))

Password(description='Pass:', layout=Layout(width='400px'))

Text(value='', description='Portal:', layout=Layout(width='500px'), placeholder='http://domain:port/c/')

Text(value='', description='MAC:', layout=Layout(width='400px'), placeholder='00:1A:79:XX:XX:XX')

Text(value='', description='M3U URL:', layout=Layout(width='600px'), placeholder='http://domain/playlist.m3u a…

HTML(value='<span style="color:#6b7280;font-size:12px;">⚡ Masukkan URL file .m3u/.m3u8 — diparse & dicek chann…

✅ IPTV Ultimate Tool v9 siap!

  🔍 Scanner  : Xtream / MAC / Raw M3U → score, alive/dead, deteksi XXX

  💾 Converter: Xtream / MAC → .m3u
     Konten   : Live | 🔞 XXX | VOD | Series
     MAC Mode : 3 pilihan kecepatan resolve URL
       • CMD langsung  → ⚡ cepat (rekomen untuk 10.000+ channel)
       • Resolve URL   → akurat tapi lambat (gunakan thread tinggi)
       • URL mentah    → paling cepat


In [3]:
import requests
import datetime
import re
import ipywidgets as widgets
from IPython.display import display, HTML
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote
import csv
import os
import threading

requests.packages.urllib3.disable_warnings()

display(HTML("""
<div style="font-family:monospace;background:#1e293b;color:#f8fafc;
            padding:14px 20px;border-radius:10px;margin-bottom:8px;">
  <span style="font-size:18px;font-weight:700;">📡 MAC Multi-Tester</span><br>
  <span style="font-size:12px;color:#94a3b8;">
    1 Portal URL → Test banyak MAC sekaligus → Temukan yang terbaik
  </span>
</div>
"""))

# ============================================================
#  KONSTANTA
# ============================================================
MAX_MAC   = 10
MAC_REGEX = re.compile(r'^([0-9A-Fa-f]{2}[:\-]){5}([0-9A-Fa-f]{2})$')
MAG_UA    = (
    "Mozilla/5.0 (QtEmbedded; U; Linux; C) AppleWebKit/533.3"
    " (KHTML, like Gecko) MAG200 stbapp ver: 2 rev: 250 Safari/533.3"
)

# ============================================================
#  UI — PORTAL
# ============================================================
portal_inp = widgets.Text(
    description='Portal URL:',
    placeholder='http://domain.com:8080',
    layout=widgets.Layout(width='550px')
)
portal_note = widgets.HTML(
    '<span style="font-size:12px;color:#6b7280;margin-left:8px;">'
    'Tanpa /c/ di akhir. Contoh: <code>http://domain.com:8080</code></span>'
)

# ============================================================
#  UI — MAC INPUTS (10 baris)
# ============================================================
mac_header = widgets.HTML("""
<div style="font-family:monospace;margin:12px 0 4px 0;">
  <b style="font-size:13px;">🔲 Daftar MAC Address (isi yang ingin ditest, maks 10):</b>
</div>
""")

mac_inputs  = []   # Text widget saja
mac_status  = []   # HTML label live status per baris
mac_rows_ui = []   # HBox per baris

for i in range(MAX_MAC):
    num_lbl = widgets.HTML(
        f'<b style="font-family:monospace;font-size:13px;'
        f'color:#64748b;min-width:28px;display:inline-block;">{i+1}.</b>'
    )
    txt = widgets.Text(
        placeholder=f'00:1A:79:XX:XX:XX',
        layout=widgets.Layout(width='230px')
    )
    slbl = widgets.HTML(
        '<span style="font-size:12px;color:#94a3b8;margin-left:10px;">—</span>'
    )
    row = widgets.HBox(
        [num_lbl, txt, slbl],
        layout=widgets.Layout(align_items='center', margin='2px 0')
    )
    mac_inputs.append(txt)
    mac_status.append(slbl)
    mac_rows_ui.append(row)

# ============================================================
#  UI — OPSI
# ============================================================
sep1 = widgets.HTML(
    '<hr style="border:0;border-top:1px solid #334155;margin:10px 0;">'
)
opt_title = widgets.HTML(
    '<b style="font-family:monospace;font-size:13px;">⚙️ Opsi Test:</b>'
)
chk_profile  = widgets.Checkbox(value=True,  description='Info profil (nama, expired)')
chk_channels = widgets.Checkbox(value=True,  description='Hitung jumlah channel')
chk_stream   = widgets.Checkbox(value=True,  description='Test 1 stream (cek streaming aktif)')
chk_export   = widgets.Checkbox(value=True,  description='Export hasil ke CSV')

parallel_sl = widgets.IntSlider(
    value=5, min=1, max=10, description='Paralel:',
    layout=widgets.Layout(width='400px')
)
parallel_note = widgets.HTML(
    '<span style="font-size:11px;color:#6b7280;">'
    'Jumlah MAC yang ditest bersamaan</span>'
)

sep2 = widgets.HTML(
    '<hr style="border:0;border-top:1px solid #334155;margin:10px 0;">'
)

btn_run   = widgets.Button(
    description="🚀 RUN TEST SEMUA MAC",
    button_style='success',
    layout=widgets.Layout(width='60%', height='40px')
)
btn_clear = widgets.Button(
    description="🗑️ Clear",
    button_style='danger',
    layout=widgets.Layout(width='18%', height='40px')
)
btn_row = widgets.HBox(
    [btn_run, btn_clear],
    layout=widgets.Layout(gap='8px')
)

progress     = widgets.IntProgress(
    value=0, min=0, max=10, description='Progress:',
    bar_style='info', layout=widgets.Layout(width='100%')
)
status_label = widgets.Label(value="✅ Siap — isi portal dan MAC lalu klik RUN")

out_visual = widgets.Output()
out_log    = widgets.Output()

res_tab = widgets.Tab(children=[out_visual, out_log])
res_tab.set_title(0, '📊 Hasil')
res_tab.set_title(1, '📋 Log')

# ============================================================
#  TAMPILKAN SEMUA WIDGET
# ============================================================
display(
    portal_inp, portal_note,
    mac_header,
    *mac_rows_ui,
    sep1,
    opt_title,
    widgets.HBox([chk_profile, chk_channels, chk_stream]),
    chk_export,
    widgets.HBox([parallel_sl, parallel_note]),
    sep2,
    btn_row,
    progress, status_label,
    res_tab
)

# ============================================================
#  FUNGSI HELPER
# ============================================================
def validate_url(url):
    try:
        from urllib.parse import urlparse
        r = urlparse(url.strip())
        return r.scheme in ("http","https") and bool(r.netloc)
    except:
        return False

def fmt_mac(mac):
    """Normalisasi MAC → uppercase, separator ':'"""
    mac = mac.strip().upper()
    mac = mac.replace('-', ':')
    return mac

def make_session(portal_val, mac_clean):
    s = requests.Session()
    s.headers.update({
        "User-Agent":       MAG_UA,
        "X-Requested-With": "XMLHttpRequest",
        "Referer":          f"{portal_val}/portal.php"
    })
    s.cookies.update({
        "mac": mac_clean, "stb_lang": "en", "timezone": "Europe/Amsterdam"
    })
    return s

def do_handshake(portal_val, session):
    resp  = session.get(
        f"{portal_val}/portal.php?type=stb&action=handshake&JsHttpRequest=1-xml",
        timeout=10, verify=False
    ).json()
    js    = resp.get("js", {})
    token = js.get("token") if isinstance(js, dict) else None
    if token:
        session.headers.update({"Authorization": f"Bearer {token}"})
    return token

def get_profile(portal_val, session):
    try:
        js = session.get(
            f"{portal_val}/portal.php?type=account_info"
            f"&action=get_main_info&JsHttpRequest=1-xml",
            timeout=10, verify=False
        ).json().get("js", {})
        name   = (js.get("fname","") + " " + js.get("lname","")).strip()
        name   = name or js.get("login","") or ""
        expire = (js.get("end_date","") or js.get("phone","") or
                  js.get("expire_date","") or "")
        status = js.get("status","") or js.get("account_status","")
        return name, expire, status
    except:
        return "", "", ""

def count_channels(portal_val, session):
    # Coba get_all_channels
    try:
        js = session.get(
            f"{portal_val}/portal.php?type=itv&action=get_all_channels&JsHttpRequest=1-xml",
            timeout=12, verify=False
        ).json().get("js", {})
        ch = js.get("data", []) if isinstance(js, dict) else []
        if ch: return len(ch)
    except:
        pass
    # Fallback: get_ordered_list → total_items
    try:
        js = session.get(
            f"{portal_val}/portal.php?type=itv&action=get_ordered_list"
            f"&genre=*&p=1&JsHttpRequest=1-xml",
            timeout=12, verify=False
        ).json().get("js", {})
        if isinstance(js, dict):
            t = js.get("total_items")
            if t: return int(t)
            return len(js.get("data", []))
    except:
        pass
    return 0

def test_stream(portal_val, session):
    try:
        js   = session.get(
            f"{portal_val}/portal.php?type=itv&action=get_ordered_list"
            f"&genre=*&p=1&JsHttpRequest=1-xml",
            timeout=10, verify=False
        ).json().get("js", {})
        data = js.get("data", []) if isinstance(js, dict) else []
        if not data: return False, ""

        cmd     = data[0].get("cmd","")
        if not cmd: return False, ""
        cmd_enc = quote(str(cmd), safe=':/?&= #')
        js2     = session.get(
            f"{portal_val}/portal.php?type=itv&action=create_link"
            f"&cmd={cmd_enc}&JsHttpRequest=1-xml",
            timeout=10, verify=False
        ).json().get("js", {})
        raw = str(js2.get("cmd","") or js2.get("url","")).strip()
        url = ""
        for part in reversed(raw.split()):
            if "://" in part:
                url = part; break
        if not url: return False, ""

        r = session.get(url, timeout=8, stream=True, verify=False,
                        headers={"Accept":"*/*","User-Agent": MAG_UA})
        return r.status_code in (200, 206), url
    except:
        return False, ""

# ============================================================
#  FUNGSI TEST SATU MAC
# ============================================================
def test_one(portal_val, mac_raw, idx,
             do_profile, do_channels, do_stream):
    mac    = fmt_mac(mac_raw)
    result = {
        "no":       idx + 1,
        "mac":      mac,
        "online":   False,
        "token":    "",
        "name":     "",
        "expire":   "",
        "acc_status": "",
        "channels": 0,
        "stream_ok": False,
        "stream_url": "",
        "ping_ms":  0,
        "error":    ""
    }

    try:
        session = make_session(portal_val, mac)

        t0    = datetime.datetime.now()
        token = do_handshake(portal_val, session)
        ping  = int((datetime.datetime.now() - t0).total_seconds() * 1000)

        result["ping_ms"] = ping
        result["token"]   = (token[:16] + "...") if token else "(no token)"
        result["online"]  = True

        if do_profile:
            name, expire, acc_status = get_profile(portal_val, session)
            result["name"]       = name
            result["expire"]     = expire
            result["acc_status"] = acc_status

        if do_channels:
            result["channels"] = count_channels(portal_val, session)

        if do_stream:
            ok, url             = test_stream(portal_val, session)
            result["stream_ok"] = ok
            result["stream_url"] = url

    except requests.exceptions.ConnectionError:
        result["error"] = "Tidak bisa konek ke server"
    except requests.exceptions.Timeout:
        result["error"] = "Timeout"
    except Exception as e:
        result["error"] = str(e)[:80]

    return result

# ============================================================
#  RENDER TABEL HASIL
# ============================================================
def render_table(results, portal_val, do_channels, do_stream):
    def expire_color(exp_str):
        if not exp_str or exp_str == "—": return "#6b7280"
        for fmt in ("%Y-%m-%d","%d-%m-%Y","%Y/%m/%d","%d/%m/%Y",
                    "%Y-%m-%d %H:%M","%d-%m-%Y %H:%M"):
            try:
                dt = datetime.datetime.strptime(exp_str.strip(), fmt)
                return "#dc2626" if dt < datetime.datetime.now() else "#16a34a"
            except:
                continue
        return "#6b7280"

    # Urutkan: online dulu, lalu sort channel terbanyak
    sorted_r = sorted(results,
                      key=lambda x: (0 if x["online"] else 1, -x["channels"]))

    rows = ""
    for r in sorted_r:
        bg    = "#f0fdf4" if r["online"] else "#fef2f2"
        st_c  = "#16a34a" if r["online"] else "#dc2626"
        st_t  = "✅ ONLINE" if r["online"] else "❌ GAGAL"
        ex_c  = expire_color(r["expire"])
        exp   = r["expire"] or "—"

        # Badge stream
        if not do_stream:
            stream_cell = '<td style="padding:6px 10px;text-align:center;color:#6b7280;">—</td>'
        elif r["stream_ok"]:
            stream_cell = '<td style="padding:6px 10px;text-align:center;color:#16a34a;font-weight:700;">✅ HIDUP</td>'
        else:
            stream_cell = '<td style="padding:6px 10px;text-align:center;color:#dc2626;font-weight:700;">❌ MATI</td>'

        ch_cell = (f'<td style="padding:6px 10px;text-align:center;font-weight:700;">'
                   f'{r["channels"] if do_channels else "—"}</td>')

        err_html = (f'<br><span style="color:#ef4444;font-size:11px;">{r["error"]}</span>'
                    if r["error"] else "")

        rows += f"""
        <tr style="background:{bg};">
          <td style="padding:6px 10px;text-align:center;color:#6b7280;">{r['no']}</td>
          <td style="padding:6px 10px;font-family:monospace;font-weight:700;font-size:13px;">
            {r['mac']}
          </td>
          <td style="padding:6px 10px;font-weight:700;color:{st_c};">{st_t}{err_html}</td>
          <td style="padding:6px 10px;">{r['name'] or '—'}</td>
          <td style="padding:6px 10px;color:{ex_c};font-weight:600;">{exp}</td>
          {ch_cell}
          {stream_cell}
          <td style="padding:6px 10px;text-align:center;color:#6b7280;font-size:12px;">
            {r['ping_ms']} ms
          </td>
        </tr>"""

    online_count = sum(1 for r in results if r["online"])
    best         = next((r for r in sorted_r if r["online"]), None)
    best_html    = ""
    if best:
        best_html = f"""
        <div style="background:#064e3b;color:#d1fae5;padding:10px 14px;
                    border-radius:8px;margin-bottom:10px;font-family:monospace;">
          🏆 <b>MAC Terbaik:</b> &nbsp;
          <span style="font-size:15px;letter-spacing:1px;">{best['mac']}</span>
          &nbsp;|&nbsp; Channel: <b>{best['channels']}</b>
          &nbsp;|&nbsp; Ping: <b>{best['ping_ms']} ms</b>
          {'&nbsp;|&nbsp; Stream: <b>✅ HIDUP</b>' if best['stream_ok'] else ''}
        </div>"""

    return f"""
    <div style="font-family:monospace;">
      <div style="margin-bottom:6px;font-size:13px;color:#6b7280;">
        Portal: <b style="color:#f8fafc;">{portal_val}</b>
      </div>
      <div style="margin-bottom:10px;font-size:13px;">
        Total MAC: <b>{len(results)}</b> &nbsp;|&nbsp;
        <span style="color:#16a34a;font-weight:700;">✅ Online: {online_count}</span>
        &nbsp;|&nbsp;
        <span style="color:#dc2626;font-weight:700;">❌ Gagal: {len(results)-online_count}</span>
      </div>
      {best_html}
      <div style="overflow-x:auto;">
      <table style="border-collapse:collapse;width:100%;font-size:13px;
                    border:1px solid #334155;min-width:750px;">
        <thead>
          <tr style="background:#1e293b;color:#f8fafc;">
            <th style="padding:8px 10px;text-align:center;width:35px;">No</th>
            <th style="padding:8px 10px;text-align:left;width:170px;">MAC Address</th>
            <th style="padding:8px 10px;text-align:left;width:100px;">Status</th>
            <th style="padding:8px 10px;text-align:left;width:120px;">Nama Akun</th>
            <th style="padding:8px 10px;text-align:left;width:110px;">Expired</th>
            <th style="padding:8px 10px;text-align:center;width:80px;">Channel</th>
            <th style="padding:8px 10px;text-align:center;width:90px;">Stream</th>
            <th style="padding:8px 10px;text-align:center;width:70px;">Ping</th>
          </tr>
        </thead>
        <tbody>{rows}</tbody>
      </table>
      </div>
    </div>"""

# ============================================================
#  EXPORT CSV
# ============================================================
def do_export_csv(results, portal_val, filename):
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(["No","MAC","Portal","Status","Nama","Expired",
                    "Acc Status","Channel","Stream","Ping (ms)","Error"])
        for r in results:
            w.writerow([
                r["no"], r["mac"], portal_val,
                "ONLINE" if r["online"] else "GAGAL",
                r["name"], r["expire"], r["acc_status"],
                r["channels"],
                "OK" if r["stream_ok"] else ("MATI" if r["stream_ok"] is False else "—"),
                r["ping_ms"], r["error"]
            ])

# ============================================================
#  MAIN — RUN TEST
# ============================================================
def run_test(b):
    out_visual.clear_output()
    out_log.clear_output()
    progress.value = 0
    status_label.value = "🔄 Memvalidasi input..."

    # --- Validasi portal ---
    portal_raw = portal_inp.value.strip().rstrip('/')
    if not validate_url(portal_raw):
        status_label.value = "❌ Portal URL tidak valid"
        with out_log:
            print("❌ Portal URL tidak valid.")
            print("   Contoh yang benar: http://domain.com:8080")
        return

    # --- Kumpulkan MAC yang diisi ---
    mac_list = []
    for i, txt in enumerate(mac_inputs):
        val = txt.value.strip()
        if not val:
            continue
        val = fmt_mac(val)
        if not MAC_REGEX.match(val):
            mac_status[i].value = (
                f'<span style="font-size:12px;color:#f59e0b;margin-left:10px;">'
                f'⚠️ Format salah</span>'
            )
            continue
        mac_list.append((i, val))

    if not mac_list:
        status_label.value = "⚠️ Tidak ada MAC valid"
        with out_log:
            print("⚠️  Tidak ada MAC address yang valid.")
            print("   Format: 00:1A:79:XX:XX:XX")
        return

    # Reset semua status label
    for _, _, slbl in [(None, None, mac_status[i]) for i, _ in mac_list]:
        slbl.value = '<span style="font-size:12px;color:#60a5fa;margin-left:10px;">⏳ Menunggu...</span>'

    do_profile   = chk_profile.value
    do_channels  = chk_channels.value
    do_stream    = chk_stream.value
    n_parallel   = parallel_sl.value

    progress.max   = len(mac_list)
    progress.value = 0
    status_label.value = f"🔄 Testing {len(mac_list)} MAC ke {portal_raw}..."

    with out_log:
        print(f"Portal  : {portal_raw}")
        print(f"Total   : {len(mac_list)} MAC | Paralel: {n_parallel}")
        print(f"Opsi    : Profil={do_profile} | Channel={do_channels} | Stream={do_stream}\n")

    results  = [None] * len(mac_list)
    done     = 0
    lock     = threading.Lock()

    def _task(job_idx, orig_idx, mac):
        r = test_one(portal_raw, mac, orig_idx,
                     do_profile, do_channels, do_stream)
        return job_idx, orig_idx, r

    with ThreadPoolExecutor(max_workers=n_parallel) as executor:
        futures = {
            executor.submit(_task, job_i, orig_i, mac): job_i
            for job_i, (orig_i, mac) in enumerate(mac_list)
        }
        for future in as_completed(futures):
            try:
                job_i, orig_i, r = future.result()
                results[job_i]   = r

                # Update status label di baris MAC
                if r["online"]:
                    ch_txt = f" | {r['channels']} ch" if do_channels else ""
                    st_txt = f" | Stream ✅" if (do_stream and r["stream_ok"]) else \
                             f" | Stream ❌" if (do_stream and not r["stream_ok"]) else ""
                    mac_status[orig_i].value = (
                        f'<span style="font-size:12px;color:#16a34a;'
                        f'font-weight:600;margin-left:10px;">'
                        f'✅ ONLINE{ch_txt}{st_txt}</span>'
                    )
                else:
                    err = f" — {r['error']}" if r["error"] else ""
                    mac_status[orig_i].value = (
                        f'<span style="font-size:12px;color:#dc2626;margin-left:10px;">'
                        f'❌ GAGAL{err}</span>'
                    )

                with out_log:
                    st = "✅ ONLINE" if r["online"] else "❌ GAGAL"
                    print(f"  [{r['mac']}] {st} | {r['ping_ms']} ms"
                          + (f" | {r['channels']} ch" if do_channels else "")
                          + (f" | Stream: {'OK' if r['stream_ok'] else 'MATI'}" if do_stream else "")
                          + (f" | ERR: {r['error']}" if r["error"] else ""))

            except Exception as ex:
                job_i = futures[future]
                with out_log:
                    print(f"  ⚠️  Error job {job_i}: {ex}")

            with lock:
                done += 1
                progress.value = done
                online_now = sum(1 for r in results if r and r["online"])
                status_label.value = (
                    f"Progress: {done}/{len(mac_list)} | ✅ Online: {online_now}"
                )

    final = [r for r in results if r is not None]

    # Render tabel visual
    with out_visual:
        display(HTML(render_table(final, portal_raw, do_channels, do_stream)))

        # Export
        if chk_export.value:
            ts    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            fname = f"mac_test_{ts}.csv"
            fpath = os.path.join(os.getcwd(), fname)
            do_export_csv(final, portal_raw, fpath)
            display(HTML(f"""
            <div style="margin-top:12px;padding:12px 14px;background:#fffbeb;
                        border:1px solid #fcd34d;border-radius:8px;
                        font-family:monospace;font-size:12px;">
              💾 <b>Hasil disimpan:</b> <code>{fname}</code><br><br>
              <b>Download di Google Colab:</b><br>
              <code>from google.colab import files; files.download('{fname}')</code>
            </div>
            """))

    online_count = sum(1 for r in final if r["online"])
    status_label.value = (
        f"✅ Selesai — {len(final)} MAC | "
        f"Online: {online_count} | Gagal: {len(final)-online_count}"
    )

    with out_log:
        print(f"\n{'='*50}")
        print(f"  Selesai: {len(final)} MAC")
        print(f"  Online : {online_count}")
        print(f"  Gagal  : {len(final)-online_count}")
        if chk_export.value:
            ts    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            fname = f"mac_test_{ts}.csv"
            print(f"\n  Download: from google.colab import files")
            print(f"            files.download('{fname}')")

# ============================================================
#  CLEAR
# ============================================================
def clear_all(b):
    portal_inp.value = ""
    for txt in mac_inputs:
        txt.value = ""
    for slbl in mac_status:
        slbl.value = '<span style="font-size:12px;color:#94a3b8;margin-left:10px;">—</span>'
    out_visual.clear_output()
    out_log.clear_output()
    progress.value = 0
    status_label.value = "✅ Siap"

btn_run.on_click(run_test)
btn_clear.on_click(clear_all)

print("✅ MAC Multi-Tester siap!")
print()
print("  Cara pakai:")
print("  1. Isi Portal URL  →  http://domain.com:8080")
print("  2. Isi MAC 1 s/d 10  →  00:1A:79:XX:XX:XX")
print("  3. Klik RUN TEST SEMUA MAC")
print()
print("  Hasil per MAC: Status | Nama | Expired | Jumlah Channel | Stream | Ping")
print("  MAC terbaik ditampilkan otomatis di bagian atas hasil")

Text(value='', description='Portal URL:', layout=Layout(width='550px'), placeholder='http://domain.com:8080')

HTML(value='<span style="font-size:12px;color:#6b7280;margin-left:8px;">Tanpa /c/ di akhir. Contoh: <code>http…

HTML(value='\n<div style="font-family:monospace;margin:12px 0 4px 0;">\n  <b style="font-size:13px;">🔲 Daftar …

HTML(value='<hr style="border:0;border-top:1px solid #334155;margin:10px 0;">')

HTML(value='<b style="font-family:monospace;font-size:13px;">⚙️ Opsi Test:</b>')

Checkbox(value=True, description='Export hasil ke CSV')

HTML(value='<hr style="border:0;border-top:1px solid #334155;margin:10px 0;">')

IntProgress(value=0, bar_style='info', description='Progress:', layout=Layout(width='100%'), max=10)

Label(value='✅ Siap — isi portal dan MAC lalu klik RUN')

✅ MAC Multi-Tester siap!

  Cara pakai:
  1. Isi Portal URL  →  http://domain.com:8080
  2. Isi MAC 1 s/d 10  →  00:1A:79:XX:XX:XX
  3. Klik RUN TEST SEMUA MAC

  Hasil per MAC: Status | Nama | Expired | Jumlah Channel | Stream | Ping
  MAC terbaik ditampilkan otomatis di bagian atas hasil
